<h2>Representation of Forward instrument via OOP</h2>

In [ ]:
import pandas as pd
import numpy as np

import random

import plotly.graph_objects as go

Consider the forward payoff function $f(S_T) = S_T - K,$ with its fair price given by $$PV = E\bigg(\frac{S_T - K}{(1+r)^{T}}\bigg).$$
Let $S_0 = 100,\: u = 1.01,\: d = 0.994,\: r = 16\%.$

And the forward contract $K = 110,\: T = 3$ months.

It is clear that since $u,d$ are measured at daily scale, and interest rate is measured at annual. To compute $p$, we need to find daily measure for risk-free interest rate.

In [ ]:
class forward():
    def __init__(self, strike, maturity):
        self.K = strike
        self.T = maturity
    
    def payoff(self, price):
        return price - self.K

In [ ]:
fwd = forward(100, 0.25) 

In [ ]:
print(fwd.K, fwd.T)
fwd.payoff(90)

In [ ]:
u = 1.01
d = 0.994
r = 0.16
r_daily = r/365

T = 90  
S_0 = 100
K = 110

In [ ]:
q = (1+r_daily-d)/(u-d)
print(q)

In [ ]:
np.prod([-2, 3, 5])

In [ ]:
class binomial_model():
        def __init__(self, up, down, rate, prob, initial):
            self.u = up
            self.d = down
            self.p = prob
            self.r = rate
            self.s0 = initial
            
        def stock_price(self, n):
            seq = np.array(random.choices([self.d, self.u], weights = [1-self.p, self.p], k = n))
            return self.s0*np.prod(seq)

In [ ]:
model = binomial_model(u, d, r_daily, q, S_0)
S_T = model.stock_price(90)
S_T

In [ ]:
fwd = forward(K, T)

In [ ]:
class pricer():
    def __init__(self, pricing_model):
        self.model = pricing_model
    
    def price(self, instrument):
        pass

In [ ]:
class monte_carlo_pricer(pricer):
    def __init__(self, model, n_paths):
        super().__init__(model)
        self.N = n_paths
        
    def price(self, instrument):
        maturity = instrument.T
        S_T = [self.model.stock_price(maturity) for _ in range(self.N)]
        payoffs = [instrument.payoff(s) for s in S_T]
        return np.mean(payoffs)/(1+self.model.r)**maturity        

In [ ]:
N = 100_000
mc_pricer = monte_carlo_pricer(model, N)
mc_pricer.price(fwd)

In [ ]:
class analytic_pricer(pricer):
    def __init__(self, model):
        super().__init__(model)
    
    def price(self, instrument):
        maturity = instrument.T
        return model.s0 - instrument.K/(1+self.model.r)**maturity

In [ ]:
an_pricer = analytic_pricer(model)
an_pricer.price(fwd)

In [ ]:
class pde_pricer(pricer):
    def __init__(self, mod):
        super().__init__(mod)
        
    def price(self, instr):
        r = self.model.r
        K = instr.K
        T = instr.T
        sig = 0.001
        s0 = self.model.s0
           
        N=2*s0
        n= 1000
        m= 5000
        
        h = N/n
        tau = T/m
        alpha = tau/h**2
        
        X = np.linspace(0.001,N,n)  
        U = np.linspace(0,N,n)
        W = np.linspace(0,N,n)
        
        for i in range(n):
            U[i] = X[i]-K
            
        t = T
        while( t > tau+1e-6 ):
           t -= tau
           for i in range(1,n-1):
               df = 1/(1+tau*r)
               D2 = (X[i]*sig)**2*tau*(U[i+1] - 2*U[i] + U[i-1])/2/h**2
               W[i] = df*( U[i] + D2 + r*X[i]*tau*(U[i+1]-U[i-1])/2/h )
           for i in range(1,n-1):
               U[i] = W[i]
        
        for i in range(1,n):
            if X[i-1] < s0 and X[i]>=s0:
                break
                
        return U[i]

In [ ]:
pde = pde_pricer(model)
pde_price = pde.price(fwd)
pde_price